# VNIR_SWIR Data analysis

#### libraries

In [10]:
import glob
import os
import pandas as pd
import numpy as np
from specdal import Spectrum
import matplotlib.pyplot as plt
import math
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler
from scipy.signal import find_peaks

In [ ]:
def load_asd_data(data_folder):
    """
    Load ASD files from the specified folder and combine into a DataFrame.
    
    Parameters:
    data_folder (str): Path to the folder containing .asd files
    
    Returns:
    pd.DataFrame: Combined DataFrame with wavelengths as index, samples as columns, and 'Avg refl' column
    """
    # Collect all .asd files
    files = sorted(glob.glob(os.path.join(data_folder, '*.asd')))

    # Initialize an empty list to hold DataFrames
    dfs = []

    # Loop over all files
    for f in files:
        try:
            # Load the spectrum
            spec = Spectrum(filepath=f)
            # Get the filename without extension for column name
            name = os.path.basename(f).replace('.asd', '')
            # Create a DataFrame from the measurement (wavelengths as index, reflectance as column)
            df = spec.measurement.to_frame(name=name)
            dfs.append(df)
        except Exception as e:
            print(f"Error loading {f}: {e}")

    # Combine all DataFrames into one, aligned by wavelength index
    if dfs:
        alldata = pd.concat(dfs, axis=1)
        # Compute average reflectance
        alldata['Avg refl'] = alldata.mean(axis=1)
        print("Data loaded successfully. Shape:", alldata.shape)
        print(alldata.head())
        return alldata
    else:
        print("No valid files loaded.")
        return None

def plot_average_spectrum(alldata):
    """
    Plot the average spectrum.
    
    Parameters:
    alldata (pd.DataFrame): DataFrame with spectral data
    """
    plt.plot(alldata.index, alldata['Avg refl'])
    plt.xlabel("Wavelength (nm)")
    plt.ylabel("Reflectance")
    plt.title("Average ASD Spectrum")
    plt.show()

def perform_clustering(alldata, num_clusters=None):
    """
    Perform hierarchical clustering on the spectral data.
    
    Parameters:
    alldata (pd.DataFrame): DataFrame with spectral data
    num_clusters (int): Number of clusters (optional, defaults to 16)
    
    Returns:
    tuple: (linkage_matrix, cluster_assignment, num_clusters, cluster_averages)
    """
    # Prepare data for clustering
    spectrum_data = alldata.drop('Avg refl', axis=1)

    # Compute pairwise distances
    distances = pdist(spectrum_data.T, metric='euclidean')
    
    # Perform hierarchical clustering
    linkage_matrix = linkage(distances, method='ward')
    
    # Determine number of clusters
    if num_clusters is None:
        num_clusters = 16  # Default value
    
    cluster_labels = fcluster(linkage_matrix, num_clusters, criterion='maxclust')

    # Create a mapping of sample to cluster
    cluster_assignment = pd.DataFrame({
        'Sample': spectrum_data.columns,
        'Cluster': cluster_labels
    })
    
    # Compute cluster averages
    cluster_averages = {}
    for cluster_id in range(1, num_clusters + 1):
        members = cluster_assignment[cluster_assignment['Cluster'] == cluster_id]['Sample'].tolist()
        cluster_data = spectrum_data[members]
        cluster_avg = cluster_data.mean(axis=1)
        cluster_averages[cluster_id] = cluster_avg
    
    return linkage_matrix, cluster_assignment, num_clusters, cluster_averages

def plot_dendrogram(linkage_matrix, spectrum_data):
    """
    Plot the hierarchical clustering dendrogram.
    
    Parameters:
    linkage_matrix: Linkage matrix from clustering
    spectrum_data (pd.DataFrame): Spectral data
    """
    plt.figure(figsize=(16, 8))
    dendrogram(linkage_matrix, labels=spectrum_data.columns.astype(str), leaf_rotation=90)
    plt.xlabel('Spectrum Sample')
    plt.ylabel('Distance (Euclidean)')
    plt.title('Hierarchical Clustering Dendrogram based on Spectral Similarity')
    plt.savefig('dendrogram.png', dpi=300, bbox_inches='tight')
    plt.show()

def plot_spectra_and_averages(spectrum_data, cluster_assignment, num_clusters, cluster_averages):
    """
    Plot all spectra colored by cluster and cluster averages.
    
    Parameters:
    spectrum_data (pd.DataFrame): Spectral data
    cluster_assignment (pd.DataFrame): Cluster assignments
    num_clusters (int): Number of clusters
    cluster_averages (dict): Cluster average spectra
    """
    # Define colors for clusters
    colors = plt.cm.viridis(np.linspace(0, 1, num_clusters))

    # All spectra colored by cluster
    plt.figure(figsize=(12, 6))
    for idx, sample in enumerate(spectrum_data.columns):
        cluster_id = cluster_assignment[cluster_assignment['Sample'] == sample]['Cluster'].values[0]
        color = colors[cluster_id - 1]
        plt.plot(spectrum_data.index, spectrum_data[sample], alpha=0.3, color=color, linewidth=0.8)
    plt.xlabel('Wavelength (nm)')
    plt.ylabel('Reflectance')
    plt.title('All Spectra Grouped by Cluster (colored)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('all_spectra_no_clusters.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Cluster averages
    plt.figure(figsize=(12, 6))
    for cluster_id in range(1, num_clusters + 1):
        cluster_avg = cluster_averages[cluster_id]
        members = cluster_assignment[cluster_assignment['Cluster'] == cluster_id]['Sample'].tolist()
        color = colors[cluster_id - 1]
        plt.plot(spectrum_data.index, cluster_avg, label=f'Cluster {cluster_id} (n={len(members)})',
                 linewidth=2.5, color=color)
    plt.xlabel('Wavelength (nm)')
    plt.ylabel('Reflectance')
    plt.title('Average Spectra per Cluster')
    plt.legend(loc='upper right', bbox_to_anchor=(1.15, 1))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('cluster_averages.png', dpi=300, bbox_inches='tight')
    plt.show()

def print_cluster_summary(cluster_assignment, num_clusters):
    """
    Print summary of cluster assignments.
    
    Parameters:
    cluster_assignment (pd.DataFrame): Cluster assignments
    num_clusters (int): Number of clusters
    """
    print("Cluster Assignment:")
    print(cluster_assignment)
    print(f"\nNumber of clusters: {num_clusters}")
    for i in range(1, num_clusters + 1):
        members = cluster_assignment[cluster_assignment['Cluster'] == i]['Sample'].tolist()
        print(f"Cluster {i}: {len(members)} samples - {members[:5]}{'...' if len(members) > 5 else ''}")
    
    print("\nCluster Summary:")
    for cluster_id in range(1, num_clusters + 1):
        members = cluster_assignment[cluster_assignment['Cluster'] == cluster_id]['Sample'].tolist()
        print(f"Cluster {cluster_id}: {len(members)} samples")

def analyze_extrema(cluster_averages, cluster_assignment):
    """
    Analyze and plot extrema (peaks and valleys) for each cluster.
    
    Parameters:
    cluster_averages (dict): Cluster average spectra
    cluster_assignment (pd.DataFrame): Cluster assignments
    """
    cluster_ids = sorted(cluster_averages.keys())
    fig, axs = plt.subplots(len(cluster_ids), 1, figsize=(14, 4 * len(cluster_ids)), sharex=True)
    if len(cluster_ids) == 1:
        axs = [axs]

    for i, cluster_id in enumerate(cluster_ids):
        cluster_avg = cluster_averages[cluster_id]
        ax = axs[i]

        # Detect local extrema on the average spectrum with filtering
        maxima_idx, _ = find_peaks(cluster_avg.values, distance=20, prominence=0.01)
        minima_idx, _ = find_peaks(-cluster_avg.values, distance=20, prominence=0.01)

        # Plot average curve
        ax.plot(cluster_avg.index, cluster_avg.values, color='tab:blue', linewidth=2, label=f'Cluster {cluster_id} avg')

        # Vertical lines: maxima / minima
        for idx in maxima_idx:
            ax.axvline(cluster_avg.index[idx], color='green', linestyle='--', alpha=0.5)
        for idx in minima_idx:
            ax.axvline(cluster_avg.index[idx], color='red', linestyle='--', alpha=0.5)

        # Mark the points
        ax.scatter(cluster_avg.index[maxima_idx], cluster_avg.values[maxima_idx], color='green', s=30, marker='^', label='max')
        ax.scatter(cluster_avg.index[minima_idx], cluster_avg.values[minima_idx], color='red', s=30, marker='v', label='min')

        ax.set_title(f'Cluster {cluster_id}\n({len(cluster_assignment[cluster_assignment.Cluster==cluster_id])} samples)')
        ax.set_ylabel('Reflectance')
        ax.legend(loc='best', fontsize='small')
        ax.grid(True, alpha=0.3)

    axs[-1].set_xlabel('Wavelength (nm)')
    plt.tight_layout()
    plt.savefig('cluster_extrema.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Optional summary table of extrema for each cluster
    for cluster_id in cluster_ids:
        cluster_avg = cluster_averages[cluster_id]
        maxima_idx, _ = find_peaks(cluster_avg.values, distance=20, prominence=0.01)
        minima_idx, _ = find_peaks(-cluster_avg.values, distance=20, prominence=0.01)

        max_points = list(zip(cluster_avg.index[maxima_idx], cluster_avg.values[maxima_idx]))
        min_points = list(zip(cluster_avg.index[minima_idx], cluster_avg.values[minima_idx]))

        print(f"\nCluster {cluster_id} extrema (count max={len(max_points)}, min={len(min_points)})")
        print("Maxima (wavelength, reflectance):")
        print(pd.DataFrame(max_points, columns=['Wavelength', 'Reflectance']).head(10).to_string(index=False))
        print("Minima (wavelength, reflectance):")
        print(pd.DataFrame(min_points, columns=['Wavelength', 'Reflectance']).head(10).to_string(index=False))

In [ ]:
# Main execution - change the data_folder to your dataset path
data_folder = 'VNIR_SWIR_LG_2024/VNIR_SWIR_LG2_024/'

# Load data
alldata = load_asd_data(data_folder)
if alldata is None:
    raise ValueError("Failed to load data")

# Plot average spectrum
plot_average_spectrum(alldata)

# Perform clustering
linkage_matrix, cluster_assignment, num_clusters, cluster_averages = perform_clustering(alldata)

# Plot dendrogram
spectrum_data = alldata.drop('Avg refl', axis=1)
plot_dendrogram(linkage_matrix, spectrum_data)

# Plot spectra and averages
plot_spectra_and_averages(spectrum_data, cluster_assignment, num_clusters, cluster_averages)

# Print cluster summary
print_cluster_summary(cluster_assignment, num_clusters)

# Analyze extrema
analyze_extrema(cluster_averages, cluster_assignment)